<a href="https://colab.research.google.com/github/codeKrantz/peptide-binding-analysis/blob/main/CritiCL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pandas numpy joblib xgboost openpyxl huggingface_hub esm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 k

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import joblib
import torch

from huggingface_hub import hf_hub_download
from google.colab import files

In [3]:
# =========================
# USER SETTINGS
# =========================
HF_REPO_ID = "KarunaAnna/CritiCL"
MODEL_FILENAME = "model_XGB.joblib"          # required
LABEL_ENCODER_FILENAME = "label_encoder.joblib"  # optional; set to None if not available
HF_REPO_TYPE = "model"                       # "model" or "dataset"
HF_REVISION = None                           # e.g. "main" or a commit hash


In [4]:
model_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=MODEL_FILENAME,
    repo_type=HF_REPO_TYPE,
    revision=HF_REVISION,
)

print("Downloaded model to:", model_path)

model = joblib.load(model_path)
print("Loaded model type:", type(model))
print("Model expects n_features =", model.n_features_in_)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_XGB.joblib:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

Downloaded model to: /root/.cache/huggingface/hub/models--KarunaAnna--CritiCL/snapshots/1a389c328df8c66fb5c7c0eb8df668bff37cb76c/model_XGB.joblib
Loaded model type: <class 'xgboost.sklearn.XGBClassifier'>
Model expects n_features = 960


/usr/lib/python3.12/pickle.py:1760: UserWarning: [01:48:07] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [5]:
label_encoder = None

if LABEL_ENCODER_FILENAME is not None:
    try:
        label_encoder_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=LABEL_ENCODER_FILENAME,
            repo_type=HF_REPO_TYPE,
            revision=HF_REVISION,
        )
        label_encoder = joblib.load(label_encoder_path)
        print("Loaded label encoder classes:", list(label_encoder.classes_))
    except Exception as e:
        print("Could not load label encoder.")
        print("Reason:", e)

label_encoder.joblib:   0%|          | 0.00/497 [00:00<?, ?B/s]

Loaded label encoder classes: ['Co', 'Ln', 'Mn', 'Ni', 'Zn']


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
def load_esmc(model_name: str = "esmc_300m", device: str = None):
    from esm.models.esmc import ESMC

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = ESMC.from_pretrained(model_name).to(device)
    model.eval()
    return model, device


model_esmc, device = load_esmc(model_name="esmc_300m")
print("ESM-C loaded on:", device)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/608 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

data/weights/esmc_300m_2024_12_v0.pth:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

ESM-C loaded on: cuda


In [7]:
@torch.no_grad()
def esmc_token_embeddings_aligned(model, sequence: str, device: str):
    """
    Mirrors your uploaded embedding script.
    Returns aligned per-residue embeddings of shape (L, d).
    """
    from esm.sdk.api import ESMProtein, LogitsConfig

    sequence = str(sequence).strip().upper()
    L = len(sequence)

    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein).to(device)

    out = model.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))
    emb = out.embeddings

    if isinstance(emb, np.ndarray):
        emb = torch.from_numpy(emb)
    emb = emb.to(device).detach()

    if emb.dim() == 3:
        if emb.shape[0] != 1:
            raise RuntimeError(f"Unexpected batch dim: {tuple(emb.shape)}")
        emb = emb.squeeze(0)

    if emb.dim() != 2:
        raise RuntimeError(f"Expected 2D embeddings (T,d), got {tuple(emb.shape)}")

    T = emb.shape[0]
    if T == L:
        return emb
    if T == L + 2:
        return emb[1:-1, :]
    if T == L + 1:
        return emb[1:, :]
    if T > L:
        start = (T - L) // 2
        return emb[start:start + L, :]

    raise RuntimeError(f"Cannot align embeddings: tokens={T}, seq_len={L}, emb_shape={tuple(emb.shape)}")


def mean_pool_residue_embeddings(emb_Ld):
    return emb_Ld.mean(dim=0)


def clean_sequence(seq):
    if pd.isna(seq):
        return ""
    return str(seq).strip().replace(" ", "").replace("\n", "").upper()


@torch.no_grad()
def sequence_to_embedding(seq, model, device):
    seq = clean_sequence(seq)
    if not seq:
        raise ValueError("Empty sequence provided.")

    emb_Ld = esmc_token_embeddings_aligned(model, seq, device=device)
    emb_d = mean_pool_residue_embeddings(emb_Ld)

    emb = emb_d.detach().cpu().numpy().astype(np.float32, copy=False)
    return emb

In [8]:
test_seq = "AKWYFGLICCKLQLK"
test_emb = sequence_to_embedding(test_seq, model_esmc, device)

print("Generated embedding dim:", test_emb.shape[0])
print("Model expected dim:", model.n_features_in_)

if test_emb.shape[0] != model.n_features_in_:
    raise ValueError(
        f"Embedding dimension mismatch: generated {test_emb.shape[0]}, "
        f"but model expects {model.n_features_in_}."
    )

Generated embedding dim: 960
Model expected dim: 960


In [9]:
def predict_from_dataframe(df, seq_col="Sequence", cycl_col="CyclizationPattern"):
    if seq_col not in df.columns:
        raise ValueError(f"Missing required sequence column: {seq_col}")

    work = df.copy()

    if cycl_col not in work.columns:
        work[cycl_col] = ""

    embeddings = []
    for seq in work[seq_col]:
        emb = sequence_to_embedding(seq, model_esmc, device)
        embeddings.append(emb)

    X = np.vstack(embeddings)

    if X.shape[1] != model.n_features_in_:
        raise ValueError(
            f"Feature shape mismatch, expected: {model.n_features_in_}, got {X.shape[1]}"
        )

    y_pred = model.predict(X)

    out = work.reset_index(drop=True).copy()

    if label_encoder is not None:
        try:
            out["prediction"] = label_encoder.inverse_transform(np.asarray(y_pred, dtype=int))
        except Exception:
            out["prediction"] = y_pred
    else:
        out["prediction"] = y_pred

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        out["confidence_max"] = proba.max(axis=1)

        if label_encoder is not None:
            class_names = list(label_encoder.classes_)
        elif hasattr(model, "classes_"):
            class_names = [str(c) for c in model.classes_]
        else:
            class_names = [f"class_{i}" for i in range(proba.shape[1])]

        for i, cname in enumerate(class_names):
            out[f"proba_{cname}"] = proba[:, i]

    return out

In [11]:
def run_single_sequence():
    seq = input("Enter sequence: ").strip()
    cyc = input("Enter cyclization pattern (blank allowed, metadata only): ").strip()

    df = pd.DataFrame([{
        "Sequence": seq,
        "CyclizationPattern": cyc
    }])

    result = predict_from_dataframe(df)
    return result

single_result = run_single_sequence()
single_result

Enter sequence: A
Enter cyclization pattern (blank allowed, metadata only): 


,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,A,,Ni,0.492964,0.131785,0.235736,0.098994,0.492964,0.040521


In [ ]:
def run_multiple_sequences():
    n = int(input("How many sequences do you want to enter? ").strip())
    rows = []

    for i in range(n):
        print(f"\nSequence {i+1}")
        seq = input("  Enter sequence: ").strip()
        cyc = input("  Enter cyclization pattern (blank allowed, metadata only): ").strip()

        rows.append({
            "Sequence": seq,
            "CyclizationPattern": cyc
        })

    df = pd.DataFrame(rows)
    result = predict_from_dataframe(df)
    return result

multi_result = run_multiple_sequences()
multi_result

How many sequences do you want to enter? 3

Sequence 1
  Enter sequence: AKLAWKPGFWYKKLAGGCAKFCCC	
  Enter cyclization pattern (blank allowed, metadata only): 19-23

Sequence 2
  Enter sequence: TPYPVNCKTDRDCVMCGLGISCKNGYCQGCT
  Enter cyclization pattern (blank allowed, metadata only): 

Sequence 3
  Enter sequence: GIPCGESCVWIPCISSAIGCSCKSKVCYRN
  Enter cyclization pattern (blank allowed, metadata only): 


,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,AKLAWKPGFWYKKLAGGCAKFCCC,19-23,Zn,0.489563,0.042744,0.029642,0.156243,0.281809,0.489563
1,TPYPVNCKTDRDCVMCGLGISCKNGYCQGCT,,Ni,0.434271,0.135465,0.019649,0.089734,0.434271,0.320881
2,GIPCGESCVWIPCISSAIGCSCKSKVCYRN,,Zn,0.635464,0.130400,0.013424,0.023955,0.196757,0.635464


In [ ]:
def run_multiple_sequences():
    n = int(input("How many sequences do you want to enter? ").strip())
    rows = []

    for i in range(n):
        print(f"\nSequence {i+1}")
        seq = input("  Enter sequence: ").strip()
        cyc = input("  Enter cyclization pattern (blank allowed, metadata only): ").strip()

        rows.append({
            "Sequence": seq,
            "CyclizationPattern": cyc
        })

    df = pd.DataFrame(rows)
    result = predict_from_dataframe(df)
    return result

multi_result = run_multiple_sequences()
multi_result

KeyboardInterrupt: Interrupted by user

In [12]:
def run_uploaded_file():
    uploaded = files.upload()
    in_file = list(uploaded.keys())[0]

    if in_file.lower().endswith(".csv"):
        df = pd.read_csv(in_file)
    elif in_file.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(in_file)
    else:
        raise ValueError("Please upload a CSV or Excel file.")

    print("Detected columns:", list(df.columns))
    seq_col = input("Enter sequence column name [default: Sequence]: ").strip() or "Sequence"
    cycl_col = input("Enter cyclization column name [default: CyclizationPattern]: ").strip() or "CyclizationPattern"

    result = predict_from_dataframe(df, seq_col=seq_col, cycl_col=cycl_col)
    return result

upload_result = run_uploaded_file()
upload_result.head()

Saving sequences_1aa.csv to sequences_1aa.csv
Detected columns: ['Sequence']
Enter sequence column name [default: Sequence]: Sequence
Enter cyclization column name [default: CyclizationPattern]: 


,Sequence,CyclizationPattern,prediction,confidence_max,proba_Co,proba_Ln,proba_Mn,proba_Ni,proba_Zn
0,A,,Ni,0.492964,0.131785,0.235736,0.098994,0.492964,0.040521
1,C,,Ln,0.528049,0.139108,0.528049,0.070278,0.233966,0.028599
2,D,,Ni,0.536975,0.137105,0.176759,0.090620,0.536975,0.058541
3,E,,Ni,0.465506,0.210629,0.177333,0.070623,0.465506,0.075910
4,F,,Ln,0.389478,0.196734,0.389478,0.092744,0.290213,0.030830


In [14]:
def save_results(df, default_name="xgb_predictions.csv"):
    out_name = input(f"Output filename [default: {default_name}]: ").strip() or default_name

    if out_name.lower().endswith(".csv"):
        df.to_csv(out_name, index=False)
    elif out_name.lower().endswith((".xlsx", ".xls")):
        df.to_excel(out_name, index=False)
    else:
        out_name += ".csv"
        df.to_csv(out_name, index=False)

    print("Saved:", out_name)
    files.download(out_name)

# Example:
# save_results(single_result)
# save_results(multi_result)
save_results(upload_result)

Output filename [default: xgb_predictions.csv]: scores_1aa
Saved: scores_1aa.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def run_menu():
    print("\nChoose input mode:")
    print("1 = Single sequence")
    print("2 = Multiple sequences manually")
    print("3 = Upload CSV/Excel")

    choice = input("Enter 1, 2, or 3: ").strip()

    if choice == "1":
        res = run_single_sequence()
    elif choice == "2":
        res = run_multiple_sequences()
    elif choice == "3":
        res = run_uploaded_file()
    else:
        raise ValueError("Invalid choice. Please enter 1, 2, or 3.")

    display(res)
    return res

results_df = run_menu()


Choose input mode:
1 = Single sequence
2 = Multiple sequences manually
3 = Upload CSV/Excel
